# LAB 12 - Random Forest for Regression

In this lab we will be extending the previous lab about Decision trees and build a Regression model using Random Forest.

For simplicity, we will be using the same dataset as the previous lab (you can find it in ECLASS).

**IMPORTANT:** For this lab, if you haven't finished your code from last week's lab on Decision trees, you will have the option to use the sklearn implementation for a regression tree. However, this doesn't mean that you should skip the previous lab. This is just so that you don't get behind with the content and you don't spend all your time today working on the previous lab. 

In [3]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split

In [4]:
housing_names = ["CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT", "MEDV"]
data = pd.read_table("housing.txt", names=housing_names, sep=r'\s+')

In [5]:
X = data.iloc[:, :-1].values
y = data["MEDV"].values


X_train, X_aux, y_train, y_aux = train_test_split(X, y, test_size=0.2)
X_val, X_test, y_val, y_test = train_test_split(X_aux, y_aux, test_size=0.5)

As mentioned before, use the Boston Housing data and prepare your train/val/test split as usual.

## Exercise 1 -- Bootstrap

Also known as [bagging](https://en.wikipedia.org/wiki/Bootstrap_aggregating), this technique consists of making several samples with replacement of the original data, using each of the samples to train an estimator, and then aggregating the predictions using the average (this is also a type of model ensemble).

In [20]:
def bootstrap(N: int, num_bags:int =10):
    """
    Given a dataset and a number of bags,
    sample the dataset with replacement.
    
    This function does not return a copy
    of the datapoints, but a list of indices
    with compatible dimensionality
    
    Parameters
    ----------
    N : ndarray
        size of the dataset
    num_bags : int, default 10
        The number of bags to create
    
    Returns
    -------
    list of ndarray
        The list contains `num_bags` integer one-dimensional ndarrays.
        Each of these contains the indices corresponding to the 
        sampled datapoints in `X`
    
    Notes
    -----
    * The number of datapoints in each bach will
      match the number of datapoints in the given
      dataset.
    * The
    """
    #N = X.shape[0]

    rng = np.random.default_rng(0) # you can change the seed, or use 0 to replicate my results
    bags = []

    for i in range(num_bags):
        bags.append(rng.choice(N, size=N))

    return bags

In [21]:
rng = np.random.default_rng(0)
X_small = rng.random(size=(100,2))
bags = bootstrap(X_small.shape[0])
bags[0]

array([85, 63, 51, 26, 30,  4,  7,  1, 17, 81, 64, 91, 50, 60, 97, 72, 63,
       54, 55, 93, 27, 81, 67,  0, 39, 85, 55,  3, 76, 72, 84, 17,  8, 86,
        2, 54,  8, 29, 48, 42, 40,  2,  0, 12,  0, 67, 52, 64, 25, 61, 76,
       38, 46, 99, 80, 98, 37, 68, 95, 65, 84, 68, 70, 38, 87, 13, 57, 72,
       84, 52, 37, 31, 42, 48, 71, 88,  7, 93, 53, 35, 67, 57, 25, 32, 71,
       59, 50, 33, 76, 39, 32, 89, 26, 22, 71, 62,  4,  8, 37, 83])

Saida esperada:
```text
array([85, 63, 51, 26, 30,  4,  7,  1, 17, 81, 64, 91, 50, 60, 97, 72, 63,
       54, 55, 93, 27, 81, 67,  0, 39, 85, 55,  3, 76, 72, 84, 17,  8, 86,
        2, 54,  8, 29, 48, 42, 40,  2,  0, 12,  0, 67, 52, 64, 25, 61, 76,
       38, 46, 99, 80, 98, 37, 68, 95, 65, 84, 68, 70, 38, 87, 13, 57, 72,
       84, 52, 37, 31, 42, 48, 71, 88,  7, 93, 53, 35, 67, 57, 25, 32, 71,
       59, 50, 33, 76, 39, 32, 89, 26, 22, 71, 62,  4,  8, 37, 83])
```

## Exercise 2 -- Aggregation

The second part of bagging.

In [ ]:
def aggregate_regression(preds: list[np.ndarray]):
    """
    Aggregate predictions by several estimators
    
    Parameters
    ----------
    preds : list of ndarray
        Predictions from multiple estimators.
        All ndarrays in this list should have the same
        dimensionality.
        
    Return
    ------
    ndarray
        The mean of the predictions
    """
    return np.mean(preds, axis=0)


## Exercise 3 -- Random Forest for regression

Using the functions you implemented above, it is now time to put all of them together to train several decision trees and then ensemble them to output a single prediction. For the random forest, however, we need to select a subset of features at each split on the decision tree. 

For this part, you can use the sklearn implementation of Decision trees for regression as your estimator for each set of features and bags. See below an example of how to do this, and always remember to check the necessary documentation when using an external function.

Some parameters you will have to set are: 
* num_features: number of features per estimator
* min_samples: min number of samples per leaf node
* max_depth: maximum depth of the decision tree (each estimator)
* num_estimators: number of decision trees you will create using each bag and random set of features

In [52]:
# your code goes here
class Node:
    def __init__(self, feature, tau, left_partition, right_partition):
        self.feature = feature
        self.tau = tau
        self.left_region = left_partition
        self.right_region = right_partition
        self.left = None
        self.right = None

class Leaf:
    def __init__(self, value):
        self.value = value

class Tree:
    def __init__(self, min_samples = 2, max_depht = None):
        self.root = None

        self.min_samples = min_samples
        self.max_depht = max_depht

    def regression_criterion(self, region: np.ndarray):
        if len(region) == 0:
            return float("inf")
        
        return np.sum((region - np.mean(region))**2)
    
    def split_region(self, region: np.ndarray, idfeature: int, tau:float):
        left_partition = region[:,idfeature] < tau
        right_partition = ~left_partition

        return left_partition, right_partition
    
    def get_split(self, X: np.ndarray, y:np.ndarray) -> dict[str, float | np.ndarray]:
        best_sse = float("inf")

        best_tau = None
        best_feature = None

        N = y.shape[0]

        # devemos ir por todas as features
        for each_feature in range(X.shape[1]):
            mask = X[:, each_feature].argsort() # retorna indices em ordem crescente

            # Acho o tau depois de achar o tau eu posso achar os indices das regioes
            X_sorted= X[mask, each_feature]
            y_sorted = y[mask] 

            for each_idy in range(1, y.shape[0]):
                
                # evitar splits impossiveis
                if X_sorted[each_idy] == X_sorted[each_idy -1]:
                    continue

                left = y_sorted[:each_idy]
                right = y_sorted[each_idy:]

                sse = (
                    self.regression_criterion(left)
                    + self.regression_criterion(right)
                )

                if sse < best_sse:
                    best_sse = sse
                    best_feature = each_feature
                    best_tau = (X_sorted[each_idy] + X_sorted[each_idy - 1]) / 2

        left_partition, right_partition =  self.split_region(X, best_feature, best_tau)

        node = Node(
            best_feature,
            best_tau,
            left_partition,
            right_partition
        )

        return node

    def recursive_growth(self, min_samples, max_depth, current_depth, X, y):

        if max_depth is not None:
            is_leaf = len(y) < min_samples or current_depth >= max_depth or self.regression_criterion(y) == 0.0
        else:
            is_leaf = len(y) < min_samples or self.regression_criterion(y) == 0.0

        if is_leaf:
            return Leaf(y.mean())
    

        node = self.get_split(X, y)

        node.left = self.recursive_growth(
            min_samples, max_depth, current_depth + 1,
            X[node.left_region],
            y[node.left_region]
        )

        node.right = self.recursive_growth(
            min_samples, max_depth, current_depth + 1,
            X[node.right_region],
            y[node.right_region]
        )

        return node
    
    def predict_sample(self, node, sample: np.ndarray):
        is_leaf = isinstance(node, Leaf)
        if is_leaf:
            return node.value
        
        if sample[node.feature] < node.tau:
            return self.predict_sample(node.left, sample)
        else: 
            return self.predict_sample(node.right, sample)
    
    def predict(self, X):
        size = X.shape[0]
        y = np.zeros(size)

        for i in range(size):
            y[i] = self.predict_sample(self.root, X[i])

        return y

    def fit(self, X, y):
        self.root = self.recursive_growth(
            self.min_samples,
            self.max_depht,
            0,
            X,
            y
        )

In [ ]:
# example of sklearn Decision tree
estimator = DecisionTreeRegressor(max_depth=max_depth)
estimator.fit(X, y)
estimator.predict(X)

In [ ]:
class RandomForest:
    def __init__(self, num_features: int, min_samples:int, max_depth:int, num_estimators:int):
        """Do a Random Forest

        Args:
            num_features (int): number of features per estimator
            min_samples (int): min number of samples per leaf node
            max_depth (int): maximum depth of the decision tree (each estimator)
            num_estimators (int): number of decision trees you will create using each bag and random set of features
        """
        self.num_features = num_features
        self.min_samples = min_samples
        self.max_depth = max_depth
        self.num_estimators = num_estimators

    def bootstrap(self, N: int):
        rng = np.random.default_rng(0) # you can change the seed, or use 0 to replicate my results
        bags = []

        for i in range(self.num_estimators):
            bags.append(rng.choice(N, size=N))

        return bags
    
    def aggregate_regression(self, preds: list[np.ndarray]):
        return np.mean(preds, axis=0)


    def fit(self, X: np.ndarray, y:np.ndarray):
        N = X.shape[0]

        bags = self.bootstrap(N)


    def predict(self, X):
        pass

In [ ]:
## your code goes here:

    estimator = Tree(5, 6)
    estimator.fit(X_train, y_train)
    estimator.predict(X_train)

array([49.85      , 19.725     , 50.        , 19.725     , 15.47837838,
       21.7483871 , 12.95      , 31.9       , 27.18333333, 15.47837838,
       21.68809524, 23.82105263, 19.725     , 19.725     , 29.62727273,
       21.7483871 , 25.375     , 17.9625    , 15.60555556, 10.975     ,
       21.7483871 , 21.68809524, 34.61428571,  7.24444444, 29.62727273,
       21.68809524, 19.78      , 21.68809524, 29.62727273, 15.47837838,
       29.62727273, 15.60555556, 21.68809524, 23.82105263, 21.68809524,
       34.61428571, 15.47837838, 23.82105263, 21.68809524, 34.61428571,
       23.82105263, 19.78      , 21.68809524, 21.7483871 , 21.7483871 ,
       21.7483871 , 17.9625    , 26.06666667, 19.725     , 15.47837838,
       40.7       , 23.82105263, 15.60555556, 21.68809524, 21.68809524,
       23.82105263, 25.375     , 21.68809524, 21.68809524,  7.24444444,
       19.725     , 29.82      , 19.725     , 49.85      ,  9.55454545,
       30.73333333, 19.725     , 19.78      , 19.78      , 21.68